In [9]:
from langgraph.graph import StateGraph, START,END 
from typing import TypedDict

from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
model_name = os.getenv("MODEL_NAME")

In [11]:
from  langchain_groq import ChatGroq
model=ChatGroq(
    api_key=api_key,
    model_name=model_name,
    temperature=0.0)

ValidationError: 1 validation error for ChatGroq
model_name
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type

In [46]:
class LLMState(TypedDict):
    question :str
    answer : str 

In [47]:
def llm_qa(state : LLMState) -> LLMState:
    #extract question
    question = state['question']

    #form a prompt 
    propmt = f"Answer the following question {question}"

    #ask question from llm 
    answer = model.invoke(propmt).content

    #update the answer
    state['answer'] = answer

    return state 

In [48]:
#create graph
graph=StateGraph(LLMState)

#add_node
graph.add_node('llm_qa',llm_qa)

graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

#compile
workflow=graph.compile()



In [49]:
#execute
init_state={"question": "How fare is moon from earth"}
final_state=workflow.invoke(init_state)
print(final_state)

{'question': 'How fare is moon from earth', 'answer': 'The distance between the Moon and the Earth is not constant because the Moon orbits the Earth in an ellipse, not a perfect circle. However, here are the key figures:\n\n- **Average distance**: Approximately **384,400 kilometers** (238,855 miles).\n- **Closest point (Perigee)**: About **363,300 kilometers** (225,700 miles).\n- **Farthest point (Apogee)**: About **405,500 kilometers** (251,900 miles).\n\nSo, when people ask "how far is the Moon from Earth," the most commonly cited answer is the **average distance of ~384,400 km (238,855 miles)**.'}
